# ASL Recognition - Colab Environment Setup
This notebook automates the process of setting up the ASL recognition environment, downloading the preprocessed frames from Box, and preparing for training.

### 1. Install Dependencies

In [ ]:
!pip install requests tqdm mediapipe opencv-python torch torchvision

### 2. Download and Unzip ASL Dataset
Downloading the **Preprocessed Frames** dataset (Top 100 Signs) directly into the Colab environment.

In [ ]:
import requests
import zipfile
import os
from tqdm import tqdm

def download_data():
    url = 'https://utdallas.box.com/s/5ry2jpk80fxvdq9xet51q65o3sd6bx0v'
    output_zip = 'asl_frames.zip'
    extract_to = 'data/frames'
    
    if not os.path.exists('data'):
        os.makedirs('data')

    print("Downloading dataset from Box...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    with open(output_zip, 'wb') as f, tqdm(
        desc=output_zip,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = f.write(data)
            bar.update(size)
    
    print("\nExtracting frames...")
    with zipfile.ZipFile(output_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    
    os.remove(output_zip)
    print(f"Success! Data extracted to {extract_to}")

download_data()

### 3. Verify Dataset Structure

In [ ]:
import os
if os.path.exists('data/frames'):
    folders = os.listdir('data/frames')
    print(f"Total labels found: {len(folders)}")
    print(f"Example labels: {folders[:10]}")

### 4. (Optional) Download Raw Videos
Run this section if you want to download the raw `.mp4` videos for the Top 100 signs. This script uses `nslt_100.json` to selectively extract only the required videos.

In [ ]:
import json

def download_raw_videos():
    # Box link for raw videos zip
    url = 'https://utdallas.box.com/s/5ry2jpk80fxvdq9xet51q65o3sd6bx0v'
    output_zip = 'raw_videos.zip'
    extract_path = 'data'
    videos_output = 'data/videos'
    subset_file = 'nslt_100.json'

    if not os.path.exists(extract_path):
        os.makedirs(extract_path)
    if not os.path.exists(videos_output):
        os.makedirs(videos_output)

    print("Downloading Raw Videos Zip from Box...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    with open(output_zip, 'wb') as f, tqdm(
        desc=output_zip,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))

    print("\nSelective Extraction starting...")
    with zipfile.ZipFile(output_zip, 'r') as z:
        # 1. Extract the JSON list first
        if subset_file in z.namelist():
            z.extract(subset_file, extract_path)
        else:
            print(f"Warning: {subset_file} not found in zip. Extracting all.")
        
        subset_json_path = os.path.join(extract_path, subset_file)
        if os.path.exists(subset_json_path):
            with open(subset_json_path, 'r') as f:
                subset_data = json.load(f)
            required_ids = set(subset_data.keys())
        else:
            required_ids = None
        
        # 2. Only extract videos that are Top 100 list
        all_files = z.namelist()
        count = 0
        for file in tqdm(all_files, desc="Filtering Videos"):
            if file.endswith('.mp4'):
                video_id = os.path.basename(file).split('.')[0]
                if required_ids is None or video_id in required_ids:
                    z.extract(file, videos_output)
                    count += 1
        
        print(f"\nSuccessfully extracted {count} videos to {videos_output}")
    
    os.remove(output_zip)

download_raw_videos()